In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
import requests
resp = requests.get('https://httpbin.org/get', timeout=10)
print(resp.status_code)

200


In [3]:
url = 'https://github.com/DataTalksClub/faq/archive/refs/heads/main.zip'
resp = requests.get(url, timeout=30)
print(resp.status_code)

200


In [4]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [5]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
print(dtc_faq[0])

{'description': 'Review and process open FAQ PRs', 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep -r

In [6]:
evidently_docs = read_repo_data('evidentlyai', 'docs')

In [7]:
print(f"Evidently docs: {len(evidently_docs)}")

Evidently docs: 95


In [8]:
doc = evidently_docs[45]
print(doc.keys())
print(f"Content length: {len(doc['content'])} characters")

dict_keys(['title', 'description', 'content', 'filename'])
Content length: 21712 characters


In [9]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [10]:
test = sliding_window(evidently_docs[45]['content'], 2000, 1000)
print(f"Number of chunks: {len(test)}")
print(f"First chunk start: {test[0]['start']}")
print(f"First chunk length: {len(test[0]['chunk'])}")

Number of chunks: 21
First chunk start: 0
First chunk length: 2000


In [11]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [12]:
# text = evidently_docs[45]['content']
# sections = split_markdown_by_level(text, level=2)
# print(f"Number of sections: {len(sections)}")
# print(sections[0])

In [13]:
# evidently_chunks = []

# for doc in evidently_docs:
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     sections = split_markdown_by_level(doc_content, level=2)
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         evidently_chunks.append(section_doc)

In [14]:
# print(f"Total chunks: {len(evidently_chunks)}")
# print(evidently_chunks[0].keys())

In [51]:
# from groq import Groq
# from dotenv import load_dotenv
# import os

# load_dotenv('../.env')

# groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# def llm(prompt, model='llama-3.3-70b-versatile'):
#     messages = [
#         {"role": "user", "content": prompt}
#     ]
    
#     response = groq_client.chat.completions.create(
#         model=model,
#         messages=messages
#     )
    
#     return response.choices[0].message.content

In [16]:
# print(llm("Say hello in one sentence."))

In [12]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

IMPORTANT: You MUST separate each section with --- on its own line. Do not skip this.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [13]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [19]:
# test_sections = intelligent_chunking(evidently_docs[45]['content'])
# print(f"Number of sections: {len(test_sections)}")
# print(test_sections[0])

In [20]:
# from tqdm.auto import tqdm

# evidently_chunks_llm = []

# for doc in tqdm(evidently_docs[:5]):  # just first 5 docs
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     sections = intelligent_chunking(doc_content)
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         evidently_chunks_llm.append(section_doc)

# print(f"Total chunks: {len(evidently_chunks_llm)}")

In [14]:
evidently_docs = read_repo_data('evidentlyai', 'docs')

evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)

In [15]:
from minsearch import Index

print(len(evidently_chunks))
print(evidently_chunks[0].keys())

576
dict_keys(['start', 'chunk', 'title', 'description', 'filename'])


In [16]:
print(len(evidently_chunks))
print(evidently_chunks[0].keys())

576
dict_keys(['start', 'chunk', 'title', 'description', 'filename'])


In [17]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)

In [18]:
query = 'What should be in a test dataset for AI evaluation?'
results = index.search(query)
print(f"Number of results: {len(results)}")
print(results[0])

Number of results: 10
{'start': 0, 'chunk': 'Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.\n\nInstead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.\n\n## Create a RAG test dataset\n\nYou can generate ground truth RAG dataset from your data source.\n\n### 1. Create a Project\n\nIn the Evidently UI, start a new Project or open an existing one.\n\n* Navigate to “Datasets” in the left menu.\n* Click “Generate” and select the “RAG” option.\n\n![](/images/synthetic/synthetic_data_select_method.png)\n\n### 2. Upload your knowledge base\n\nSelect a file containing the information your AI system retrieves from. Supported formats: Markdown (.md), CSV, TXT, PDFs. Choose how many inputs to generate.\n\n![](/images/synthetic/synthe

In [19]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

In [20]:
faq_index = Index(
    text_fields=['question','content'],  # what fields does FAQ have?
    keyword_fields=[]
)
faq_index.fit(de_dtc_faq)

In [21]:
results = faq_index.search('Can I join the course now?')
print(f"Results: {len(results)}")
print(results[0])

Results: 10
{'id': '3f1424af17', 'question': 'Course: Can I still join the course after the start date?', 'sort_order': 3, 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}


In [22]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [23]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)

similarity = v_query.dot(v_doc)
print(f"Vector size: {v_doc.shape}")
print(f"Similarity: {similarity}")

Vector size: (768,)
Similarity: 0.5190930962562561


In [24]:
print(record)

{'id': '3f1424af17', 'question': 'Course: Can I still join the course after the start date?', 'sort_order': 3, 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}


In [25]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for record in de_dtc_faq:
    text = record['question'] + ' ' + record['content']
    v_doc = embedding_model.encode(text)
    faq_embeddings.append(v_doc)

faq_embeddings = np.array(faq_embeddings)

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)

In [26]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)
print(f"Results: {len(results)}")
print(results[0])

Results: 10
{'id': '3f1424af17', 'question': 'Course: Can I still join the course after the start date?', 'sort_order': 3, 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}


In [27]:
query = 'I just discovered the program, can I still enroll?'

# text search
text_results = faq_index.search(query)
print("TEXT SEARCH:")
print(text_results[0]['question'])

# vector search
q = embedding_model.encode(query)
vector_results = faq_vindex.search(q)
print("\nVECTOR SEARCH:")
print(vector_results[0]['question'])

TEXT SEARCH:
Course: Can I still join the course after the start date?

VECTOR SEARCH:
Course: Can I still join the course after the start date?


In [28]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Search the course FAQ database for relevant information.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results from the FAQ index.
    """
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results


In [29]:
query = 'Can I join the course now?'

print(f"Text search: {len(text_search(query))} results")
print(f"Vector search: {len(vector_search(query))} results")
print(f"Hybrid search: {len(hybrid_search(query))} results")

Text search: 5 results
Vector search: 5 results
Hybrid search: 6 results


In [32]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv('../.env')
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [33]:
text_search_tool = {
    "type": "function",
    "function": {
        "name": "text_search",
        "description": "Search the FAQ database",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}


In [34]:
system_prompt = """
You are a helpful assistant for a course. 
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

In [35]:
import json

# Step 1: Initial call
response = groq_client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=chat_messages,
    tools=[text_search_tool],
    tool_choice='auto',
    temperature=0.5
)

# Step 2: Get the tool call
response_message = response.choices[0].message
print(response_message.tool_calls)

# Step 3: Execute the tool
tool_call = response_message.tool_calls[0]
function_args = json.loads(tool_call.function.arguments)
result = text_search(**function_args)

# Step 4: Append assistant message + tool result
chat_messages.append(response_message)
chat_messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "name": tool_call.function.name,
    "content": json.dumps(result)
})

# Step 5: Get final response
final_response = groq_client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=chat_messages,
    temperature=0.5
)

print(final_response.choices[0].message.content)

[ChatCompletionMessageToolCall(id='mtvyewdd8', function=Function(arguments='{"query":"late enrollment policy"}', name='text_search'), type='function')]
It seems you've just discovered the course and are wondering if you can join now. The answer is yes, you can join the course, but please note that late submissions of homework are not allowed. However, if the form is still open after the due date, you can still submit the homework. 

Also, keep in mind that you don't need to do the homework to get the certificate, as long as you complete the peer-reviewed capstone projects on time. If you have any other questions or need further assistance, feel free to ask.


In [38]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

model = GroqModel('llama-3.1-8b-instant')

system_prompt = """
You are a helpful assistant for a course. 
Use the search tool to find relevant information before answering questions.
If the first search doesn't give enough information, try different search terms.
"""

agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[text_search]
)

In [39]:
question = "I just discovered the course, can I join now?"
result = await agent.run(user_prompt=question)
print(result.output)

You can still join the course after it starts, but keep in mind that there will be deadlines for turning in homeworks and the final projects. It's not too late to join the course, but I recommend starting by installing and setting up all the dependencies and requirements, and looking over the prerequisites and syllabus to see if you are comfortable with these subjects.


In [40]:
for message in result.new_messages():
    print(message)
    print("---")

ModelRequest(parts=[SystemPromptPart(content="\nYou are a helpful assistant for a course. \nUse the search tool to find relevant information before answering questions.\nIf the first search doesn't give enough information, try different search terms.\n", timestamp=datetime.datetime(2026, 3, 27, 23, 50, 49, 152233, tzinfo=datetime.timezone.utc)), UserPromptPart(content='I just discovered the course, can I join now?', timestamp=datetime.datetime(2026, 3, 27, 23, 50, 49, 152241, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 27, 23, 50, 49, 152595, tzinfo=datetime.timezone.utc), run_id='f594c37d-ce21-4d3b-98a9-bca58abacab9')
---
ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query":"join course now"}', tool_call_id='wsgb0ttxm')], usage=RequestUsage(input_tokens=392, output_tokens=17), model_name='llama-3.1-8b-instant', timestamp=datetime.datetime(2026, 3, 27, 23, 50, 49, 384206, tzinfo=datetime.timezone.utc), provider_name='groq', provider_url='h

In [41]:
question = "can I join late and get a certificate?"
result = await agent.run(user_prompt=question)
print(result.output)

You can join late and receive a certificate, but you must complete the peer-reviewed capstone projects on time to qualify for the certificate. You do not need to do the homeworks if you join late.


In [53]:
from pydantic_ai.messages import ModelMessagesTypeAdapter


def log_entry(agent, messages, source="user"):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    # extract system prompt from first message if agent._instructions is empty
    system_prompt = agent._instructions
    if not system_prompt:
        try:
            system_prompt = dict_messages[0]['parts'][0]['content']
        except:
            system_prompt = ""

    return {
        "agent_name": agent.name,
        "system_prompt": system_prompt,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }

In [54]:
import json
import secrets
from pathlib import Path
from datetime import datetime


LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    ts = entry['messages'][-1]['timestamp']
    # ts_obj = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    ts_str = ts.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath


In [48]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())


 how do I use docker on windows?


To use Docker on Windows, first ensure that you are running the latest version of Docker for Windows. If the upgrade option in the menu doesn't work, uninstall and reinstall with the latest version. If Docker is stuck on starting, try switching the containers by right-clicking the Docker symbol from the running programs, and switch the containers from Windows to Linux or vice versa.

For Windows 10 / 11 Pro Edition:

- **Hyper-V Backend:** Ensure Hyper-V is enabled by following this tutorial.
- **WSL2 Backend:** Follow the steps detailed in this tutorial.

You may encounter this error: "The input device is not a TTY (Docker run for Windows)" when using `docker run -it ubuntu bash`. To resolve it, use `winpty` before the Docker command: `$ winpty docker run -it ubuntu bash`. Alternatively, create an alias: `echo 'alias docker='winpty docker'' >> ~/.bashrc` or `echo 'alias docker='winpty docker'' >> ~/.bash_profile`.

If you encounter the error "docker: error during connect: In the defau

PosixPath('logs/agent_20260328_000904_b92113.json')

In [58]:
import os
for f in LOG_DIR.iterdir():
    print(f)

logs/agent_20260328_001353_df78ae.json
logs/agent_20260328_001351_e4eee3.json
logs/agent_20260328_000904_b92113.json
logs/agent_20260328_001352_7f1446.json


In [57]:
def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

log_record = load_log_file(list(LOG_DIR.iterdir())[-1])
print(json.dumps(log_record, indent=2, default=str)[:500])

{
  "agent_name": "agent",
  "system_prompt": "\nYou are a helpful assistant for a course. \nUse the search tool to find relevant information before answering questions.\nIf the first search doesn't give enough information, try different search terms.\n",
  "provider": "groq",
  "model": "llama-3.1-8b-instant",
  "tools": [
    "text_search"
  ],
  "messages": [
    {
      "parts": [
        {
          "content": "\nYou are a helpful assistant for a course. \nUse the search tool to find releva


In [55]:
questions = [
    "can I join late and get a certificate?",
    "what do I need to do for the certificate?",
    "how do I install Kafka in Python?"
]

for q in questions:
    print(q)
    result = await agent.run(user_prompt=q)
    print(result.output)
    log_interaction_to_file(agent, result.new_messages())
    print()

can I join late and get a certificate?
Yes, you can join late and get a certificate. However, you need to complete the peer-reviewed capstone projects on time to receive the certificate, regardless of whether you join the course early or late. Additionally, you cannot submit homework late, but if the form is still open after the due date, you can still submit the homework.

what do I need to do for the certificate?
Based on the search results, it appears that to receive a certificate, you need to finish the course with a "live" cohort and complete the peer-reviewed capstone projects on time. You do not need to do the homeworks to get the certificate. However, you must have your full name displayed correctly on the Certificate, which can be edited on the Course Management webpage.

After completing the course, there will be an announcement in Telegram and the course channel when the grading is completed and you can generate the Certificate document yourself by following the instructions